In [11]:
import numpy as np
import joblib
import os
from lifelines.utils import concordance_index

# Settings
result_dir = '/data4/meerak/support50_propbin_data'
preds_dir = '/data4/meerak/support50_propbin_test_preds'

models = ['icind', 'pcind', 'dhcind', 'proposed_smallweight_linear', 'proposed_smallweight_exp', 'proposed_largeweight_linear', 'proposed_largeweight_exp']
model_names = ['IPCW', "Standard\nMargin", 'Standard', 'Proposed - Linear, Small Weight', 'Proposed - Exponential, Small Weight', 'Proposed - Linear, Large Weight', 'Proposed - Exponential, Large Weight']

suffixes = ['covariate_high_resource']  # You can replace with your full suffix list

def partial_cindex(y_true, y_pred, group_mask):
    group_idx = np.where(group_mask)[0]

    y_true_i = y_true[group_idx][:, np.newaxis]
    y_pred_i = y_pred[group_idx][:, np.newaxis]

    y_true_j = y_true[np.newaxis, :]
    y_pred_j = y_pred[np.newaxis, :]

    # Ignore comparisons where i == j
    mask_diff = group_idx[:, np.newaxis] != np.arange(len(y_true))

    # Valid comparisons (different y_true)
    mask_valid = (y_true_i != y_true_j) & mask_diff

    # Concordant pairs
    concordant = (
        ((y_true_i <= y_true_j) & (y_pred_i <= y_pred_j)) |
        ((y_true_i >= y_true_j) & (y_pred_i >= y_pred_j))
    ) & mask_valid

    permissible = np.sum(mask_valid)
    concordant_count = np.sum(concordant) 

    return concordant_count / permissible if permissible > 0 else np.nan

for suffix in suffixes:
    print(f"\n==== {suffix} ====")

    # Load ground truth
    orig_y_test = joblib.load(os.path.join(result_dir, f'orig_y_test_{suffix}.joblib'))
    binary_y_test = joblib.load(os.path.join(result_dir, f'binary_y_test_{suffix}.joblib'))  # 1 = event, 0 = censored

    # Define masks
    mask_lt20 = orig_y_test < 20
    mask_ge20 = orig_y_test >= 20
    mask_event = binary_y_test == 1
    mask_cens = binary_y_test == 0

    masks = {
        "<20 cens.":   mask_lt20 & mask_cens,
        "<20 uncens.": mask_lt20 & mask_event,
        "≥20 cens.":   mask_ge20 & mask_cens,
        "≥20 uncens.": mask_ge20 & mask_event,
    }

    for model, name in zip(models, model_names):
        overall_cindex_vals = []
        partial_vals = {k: [] for k in masks}

        for splitidx in range(1000):
            pred_path = os.path.join(preds_dir, f'{model}_y_test_pred_split{splitidx}_{suffix}.joblib')
            preds = joblib.load(pred_path)

            # Overall
            c_all = concordance_index(orig_y_test, preds, np.ones_like(orig_y_test))
            overall_cindex_vals.append(c_all)

            # Partial C-index for each subgroup vs. full population
            for group_name, mask in masks.items():
                c_partial = partial_cindex(orig_y_test, preds, mask)
                partial_vals[group_name].append(c_partial)

        # Summarize
        overall_cindex_vals = np.array(overall_cindex_vals)
        print(f"\n{name}")
        print(f"  Overall:       {np.median(overall_cindex_vals):.3f} [{np.percentile(overall_cindex_vals, 2.5):.3f}, {np.percentile(overall_cindex_vals, 97.5):.3f}]")
        
        for group_name in masks:
            vals = np.array(partial_vals[group_name])
            print(f"  {group_name:13s}: {np.median(vals):.3f} [{np.percentile(vals, 2.5):.3f}, {np.percentile(vals, 97.5):.3f}]")



==== covariate_high_resource ====

IPCW
  Overall:       0.605 [0.562, 0.637]
  <20 cens.    : 0.611 [0.560, 0.649]
  <20 uncens.  : 0.654 [0.613, 0.687]
  ≥20 cens.    : 0.644 [0.605, 0.670]
  ≥20 uncens.  : 0.658 [0.611, 0.698]

Standard
Margin
  Overall:       0.507 [0.444, 0.589]
  <20 cens.    : 0.679 [0.545, 0.974]
  <20 uncens.  : 0.686 [0.522, 0.978]
  ≥20 cens.    : 0.688 [0.565, 0.987]
  ≥20 uncens.  : 0.701 [0.540, 0.987]

Standard
  Overall:       0.523 [0.483, 0.567]
  <20 cens.    : 0.562 [0.514, 0.607]
  <20 uncens.  : 0.645 [0.624, 0.663]
  ≥20 cens.    : 0.606 [0.566, 0.654]
  ≥20 uncens.  : 0.490 [0.442, 0.543]

Proposed - Linear, Small Weight
  Overall:       0.641 [0.615, 0.662]
  <20 cens.    : 0.658 [0.630, 0.682]
  <20 uncens.  : 0.681 [0.639, 0.715]
  ≥20 cens.    : 0.678 [0.652, 0.700]
  ≥20 uncens.  : 0.689 [0.649, 0.730]

Proposed - Exponential, Small Weight
  Overall:       0.643 [0.590, 0.680]
  <20 cens.    : 0.711 [0.626, 0.780]
  <20 uncens.  : 0.663 [0